In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Introduction](introyt1_tutorial.html) \|\| **Tensors** \|\|
[Autograd](autogradyt_tutorial.html) \|\| [Building
Models](modelsyt_tutorial.html) \|\| [TensorBoard
Support](tensorboardyt_tutorial.html) \|\| [Training
Models](trainingyt.html) \|\| [Model Understanding](captumyt.html)

Introduction to PyTorch Tensors
===============================

Follow along with the video below or on
[youtube](https://www.youtube.com/watch?v=r7QDUPb2dCM).



In [2]:
# Run this cell to load the video
from IPython.display import display, HTML
html_code = """
<div style="margin-top:10px; margin-bottom:10px;">
  <iframe width="560" height="315" src="https://www.youtube.com/embed/r7QDUPb2dCM" frameborder="0" allow="accelerometer; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>
</div>
"""
display(HTML(html_code))



Tensors are the central data abstraction in PyTorch. This interactive
notebook provides an in-depth introduction to the `torch.Tensor` class.

First things first, let's import the PyTorch module. We'll also add
Python's math module to facilitate some of the examples.


In [1]:
import torch
import math

Creating Tensors
================

The simplest way to create a tensor is with the `torch.empty()` call:


In [6]:
x = torch.empty(3, 4)  # Crea un tensor 2D de tamaño 3x4 sin inicializar (contenido de memoria no definido).
print(type(x))  # Muestra el tipo del objeto creado (normalmente torch.Tensor).
print(x,"\n")  # Imprime los valores actuales del tensor; pueden parecer aleatorios al no haberse inicializado.

y = torch.empty(2, 3, 4)  # Crea un tensor 3D de tamaño 2x3x4 sin inicializar. 2 veces un tensor de (3,4)
print(type(y))  # Verifica que y también es un tensor de PyTorch.
print(y)  # Muestra su contenido en 3 dimensiones; valores no inicializados.

<class 'torch.Tensor'>
tensor([[2.5603e-33, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [8.9683e-44, 0.0000e+00, 1.1210e-43, 0.0000e+00],
        [4.4288e-33, 0.0000e+00, 2.0000e+00, 5.0915e-01]]) 

<class 'torch.Tensor'>
tensor([[[9.5399e-10, 4.5608e-41, 9.5399e-10, 4.5608e-41],
         [4.4842e-44, 0.0000e+00, 8.9683e-44, 0.0000e+00],
         [2.5623e-33, 0.0000e+00, 2.9907e+21, 1.0267e-08]],

        [[1.0301e-11, 6.5565e-10, 3.3611e+21, 4.2882e-08],
         [1.2780e+22, 1.0616e-08, 6.7707e+22, 3.0524e-18],
         [3.1360e+27, 7.0800e+31, 9.1084e-44, 0.0000e+00]]])


Let's upack what we just did:

-   We created a tensor using one of the numerous factory methods
    attached to the `torch` module.
-   The tensor itself is 2-dimensional, having 3 rows and 4 columns.
-   The type of the object returned is `torch.Tensor`, which is an alias
    for `torch.FloatTensor`; by default, PyTorch tensors are populated
    with 32-bit floating point numbers. (More on data types below.)
-   You will probably see some random-looking values when printing your
    tensor. The `torch.empty()` call allocates memory for the tensor,
    but does not initialize it with any values - so what you're seeing
    is whatever was in memory at the time of allocation.

A brief note about tensors and their number of dimensions, and
terminology:

-   You will sometimes see a 1-dimensional tensor called a *vector.*
-   Likewise, a 2-dimensional tensor is often referred to as a *matrix.*
-   Anything with more than two dimensions is generally just called a
    tensor.

More often than not, you'll want to initialize your tensor with some
value. Common cases are all zeros, all ones, or random values, and the
`torch` module provides factory methods for all of these:


In [7]:
zeros = torch.zeros(2, 3)
print(zeros)

ones = torch.ones(2, 3)
print(ones)

torch.manual_seed(1729)
random = torch.rand(2, 3)
print(random)

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691]])


The factory methods all do just what you'd expect - we have a tensor
full of zeros, another full of ones, and another with random values
between 0 and 1.

Random Tensors and Seeding
==========================

Speaking of the random tensor, did you notice the call to
`torch.manual_seed()` immediately preceding it? Initializing tensors,
such as a model's learning weights, with random values is common but
there are times - especially in research settings - where you'll want
some assurance of the reproducibility of your results. Manually setting
your random number generator's seed is the way to do this. Let's look
more closely:


In [12]:
print(f"Inicializamos la semilla de aleatorios con el número 1729:")
torch.manual_seed(1729)
random1 = torch.rand(2, 3)
print(random1)

random2 = torch.rand(2, 3)
print(random2)

print(f"\nVolvemos a inicializar la semilla de aleatorios con el número 1729. Los aleatorios serán iguales:")
torch.manual_seed(1729)
random3 = torch.rand(2, 3)
print(random3)

random4 = torch.rand(2, 3)
print(random4)

Inicializamos la semilla de aleatorios con el número 1729:
tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691]])
tensor([[0.2332, 0.4047, 0.2162],
        [0.9927, 0.4128, 0.5938]])

Volvemos a inicializar la semilla de aleatorios con el número 1729. Los aleatorios serán iguales:
tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691]])
tensor([[0.2332, 0.4047, 0.2162],
        [0.9927, 0.4128, 0.5938]])


What you should see above is that `random1` and `random3` carry
identical values, as do `random2` and `random4`. Manually setting the
RNG's seed resets it, so that identical computations depending on random
number should, in most settings, provide identical results.

For more information, see the [PyTorch documentation on
reproducibility](https://pytorch.org/docs/stable/notes/randomness.html).

Tensor Shapes
=============

Often, when you're performing operations on two or more tensors, they
will need to be of the same *shape* - that is, having the same number of
dimensions and the same number of cells in each dimension. For that, we
have the `torch.*_like()` methods:


In [14]:
x = torch.empty(2, 2, 3)  # Tensor base 3D de forma (2, 2, 3), sin inicializar.
print(x.shape)  # Muestra la forma del tensor base.
print(x)  # Imprime su contenido actual (valores no inicializados).

print("\n")
empty_like_x = torch.empty_like(x)  # Crea otro tensor sin inicializar con la misma forma y dtype que x.
print(empty_like_x.shape)  # Verifica que conserva la forma de x.
print(empty_like_x)  # Muestra valores no inicializados.

print("\n")
zeros_like_x = torch.zeros_like(x)  # Crea un tensor de ceros con misma forma y dtype que x.
print(zeros_like_x.shape)  # Verifica la forma.
print(zeros_like_x)  # Muestra todos los elementos en 0.

print("\n")
ones_like_x = torch.ones_like(x)  # Crea un tensor de unos con misma forma y dtype que x.
print(ones_like_x.shape)  # Verifica la forma.
print(ones_like_x)  # Muestra todos los elementos en 1.

print("\n")
rand_like_x = torch.rand_like(x)  # Crea un tensor aleatorio uniforme en [0, 1) con misma forma y dtype que x.
print(rand_like_x.shape)  # Verifica la forma.
print(rand_like_x)  # Muestra valores aleatorios.

torch.Size([2, 2, 3])
tensor([[[9.5399e-10, 4.5608e-41, 1.8445e-36],
         [0.0000e+00, 5.0286e-33, 0.0000e+00]],

        [[5.0282e-33, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 5.0141e-33, 0.0000e+00]]])


torch.Size([2, 2, 3])
tensor([[[9.5399e-10, 4.5608e-41, 4.4097e-33],
         [0.0000e+00, 4.4136e-33, 0.0000e+00]],

        [[5.0287e-33, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 4.4070e-33, 0.0000e+00]]])


torch.Size([2, 2, 3])
tensor([[[0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.]]])


torch.Size([2, 2, 3])
tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.]]])


torch.Size([2, 2, 3])
tensor([[[0.5062, 0.8469, 0.2588],
         [0.2707, 0.4115, 0.6839]],

        [[0.0703, 0.5105, 0.9451],
         [0.2359, 0.1979, 0.3327]]])


The first new thing in the code cell above is the use of the `.shape`
property on a tensor. This property contains a list of the extent of
each dimension of a tensor - in our case, `x` is a three-dimensional
tensor with shape 2 x 2 x 3.

Below that, we call the `.empty_like()`, `.zeros_like()`,
`.ones_like()`, and `.rand_like()` methods. Using the `.shape` property,
we can verify that each of these methods returns a tensor of identical
dimensionality and extent.

The last way to create a tensor that will cover is to specify its data
directly from a PyTorch collection:


In [ ]:
some_constants = torch.tensor([[3.1415926, 2.71828], [1.61803, 0.0072897]])  # Crea un tensor 2x2 a partir de valores decimales (constantes matemáticas).
print(some_constants)  # Muestra el tensor de constantes.

some_integers = torch.tensor((2, 3, 5, 7, 11, 13, 17, 19))  # Crea un tensor 1D (vector) con números enteros.
print(some_integers)  # Muestra el vector de enteros.

"""
Para torch.tensor(...) lo importante no es si usas tupla () o lista [], sino la estructura anidada y los valores.
En estos 3 casos, PyTorch ve exactamente esto:

2 filas
3 columnas
mismos números en cada posición

"""
more_integers = torch.tensor(((2, 4, 6), [3, 6, 9]))  # Crea un tensor 2D a partir de colecciones anidadas (tupla/lista).
print(more_integers)  # Muestra la matriz de enteros resultante.

prueba_aux = torch.tensor([(2, 4, 6), [3, 6, 9]]) 
print(prueba_aux)  # Muestra la matriz de enteros resultante.

prueba_aux2 = torch.tensor([[2, 4, 6], [3, 6, 9]])  
print(prueba_aux2)  # Muestra la matriz de enteros resultante.

tensor([[3.1416, 2.7183],
        [1.6180, 0.0073]])
tensor([ 2,  3,  5,  7, 11, 13, 17, 19])
tensor([[2, 4, 6],
        [3, 6, 9]])
tensor([[2, 4, 6],
        [3, 6, 9]])
tensor([[2, 4, 6],
        [3, 6, 9]])


Using `torch.tensor()` is the most straightforward way to create a
tensor if you already have data in a Python tuple or list. As shown
above, nesting the collections will result in a multi-dimensional
tensor.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p><code>torch.tensor()</code> creates a copy of the data.</p>

</div>

Tensor Data Types
=================

Setting the datatype of a tensor is possible a couple of ways:


In [ ]:
a = torch.ones((2, 3), dtype=torch.int16)  # Crea un tensor 2x3 de unos con tipo entero de 16 bits.
print(a)  # Muestra el tensor y su dtype (int16).

b = torch.rand((2, 3), dtype=torch.float64) * 20.  # Genera valores aleatorios float64 en [0,1) y los escala al rango [0,20).
print(b)  # Muestra el tensor en coma flotante de 64 bits.

c = b.to(torch.int32)  # Convierte b a enteros de 32 bits (trunca la parte decimal).
print(c)  # Muestra el tensor convertido a int32.

tensor([[1, 1, 1],
        [1, 1, 1]], dtype=torch.int16)
tensor([[18.3283,  0.2118, 18.4972],
        [ 9.8370,  3.8937, 16.1945]], dtype=torch.float64)
tensor([[18,  0, 18],
        [ 9,  3, 16]], dtype=torch.int32)


The simplest way to set the underlying data type of a tensor is with an
optional argument at creation time. In the first line of the cell above,
we set `dtype=torch.int16` for the tensor `a`. When we print `a`, we can
see that it's full of `1` rather than `1.` - Python's subtle cue that
this is an integer type rather than floating point.

Another thing to notice about printing `a` is that, unlike when we left
`dtype` as the default (32-bit floating point), printing the tensor also
specifies its `dtype`.

You may have also spotted that we went from specifying the tensor's
shape as a series of integer arguments, to grouping those arguments in a
tuple. This is not strictly necessary - PyTorch will take a series of
initial, unlabeled integer arguments as a tensor shape - but when adding
the optional arguments, it can make your intent more readable.

The other way to set the datatype is with the `.to()` method. In the
cell above, we create a random floating point tensor `b` in the usual
way. Following that, we create `c` by converting `b` to a 32-bit integer
with the `.to()` method. Note that `c` contains all the same values as
`b`, but truncated to integers.

For more information, see the [data types
documentation](https://pytorch.org/docs/stable/tensor_attributes.html#torch.dtype).

Math & Logic with PyTorch Tensors
=================================

Now that you know some of the ways to create a tensor... what can you do
with them?

Let's look at basic arithmetic first, and how tensors interact with
simple scalars:


In [ ]:
ones = torch.zeros(2, 2) + 1  # Crea un tensor 2x2 de ceros y le suma 1 a cada elemento -> quedan todos en 1.
twos = torch.ones(2, 2) * 2  # Crea un tensor 2x2 de unos y multiplica cada elemento por 2.
threes = (torch.ones(2, 2) * 7 - 1) / 2  # Aplica operaciones encadenadas elemento a elemento: ((1*7)-1)/2 = 3.
fours = twos ** 2  # Eleva al cuadrado cada elemento de 'twos' (2^2 = 4).
sqrt2s = twos ** 0.5  # Calcula la raíz cuadrada de cada elemento de 'twos' (sqrt(2)).

print(ones)  # Muestra el tensor con unos.
print(twos)  # Muestra el tensor con doses.
print(threes)  # Muestra el tensor con treses.
print(fours)  # Muestra el tensor con cuatros.
print(sqrt2s)  # Muestra el tensor con la raíz cuadrada de 2 en cada posición.

tensor([[1., 1.],
        [1., 1.]])
tensor([[2., 2.],
        [2., 2.]])
tensor([[3., 3.],
        [3., 3.]])
tensor([[4., 4.],
        [4., 4.]])
tensor([[1.4142, 1.4142],
        [1.4142, 1.4142]])


As you can see above, arithmetic operations between tensors and scalars,
such as addition, subtraction, multiplication, division, and
exponentiation are distributed over every element of the tensor. Because
the output of such an operation will be a tensor, you can chain them
together with the usual operator precedence rules, as in the line where
we create `threes`.

Similar operations between two tensors also behave like you'd
intuitively expect:


In [ ]:
#                       --- MÁS OPERACIONES CON TENSORES ---
powers2 = twos ** torch.tensor([[1, 2], [3, 4]])  # Eleva cada elemento de 'twos' según el exponente en la misma posición (potenciación elemento a elemento).
print(powers2)  # Muestra el resultado de las potencias por posición.

fives = ones + fours  # Suma tensors del mismo tamaño elemento a elemento: 1 + 4 = 5 en cada celda.
print(fives)  # Muestra el tensor resultante con cincos.

dozens = threes * fours  # Multiplica elemento a elemento: 3 * 4 = 12 en cada posición.
print(dozens)  # Muestra el tensor resultante con doces.

tensor([[ 2.,  4.],
        [ 8., 16.]])
tensor([[5., 5.],
        [5., 5.]])
tensor([[12., 12.],
        [12., 12.]])


It's important to note here that all of the tensors in the previous code
cell were of identical shape. What happens when we try to perform a
binary operation on tensors if dissimilar shape?

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>The following cell throws a run-time error. This is intentional.<pre><code>a = torch.rand(2, 3)
b = torch.rand(3, 2)</p>
<p>print(a * b)</code></pre></p>

</div>



In the general case, you cannot operate on tensors of different shape
this way, even in a case like the cell above, where the tensors have an
identical number of elements.

In Brief: Tensor Broadcasting
=============================

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>If you are familiar with broadcasting semantics in NumPyndarrays, you’ll find the same rules apply here.</p>

</div>

The exception to the same-shapes rule is *tensor broadcasting.* Here's
an example:


In [21]:
rand = torch.rand(2, 4)  # Crea un tensor aleatorio de forma (2, 4) con valores uniformes en [0, 1).
doubled = rand * (torch.ones(1, 4) * 2)  # Multiplica 'rand' por un tensor (1,4) de doses; se aplica broadcasting por filas.

print(rand)  # Muestra el tensor original.
print(doubled)  # Muestra el tensor resultante con valores duplicados.

tensor([[0.2024, 0.5731, 0.7191, 0.4067],
        [0.7301, 0.6276, 0.7357, 0.0381]])
tensor([[0.4049, 1.1461, 1.4382, 0.8134],
        [1.4602, 1.2551, 1.4715, 0.0762]])


What's the trick here? How is it we got to multiply a 2x4 tensor by a
1x4 tensor?

Broadcasting is a way to perform an operation between tensors that have
similarities in their shapes. In the example above, the one-row,
four-column tensor is multiplied by *both rows* of the two-row,
four-column tensor.

This is an important operation in Deep Learning. The common example is
multiplying a tensor of learning weights by a *batch* of input tensors,
applying the operation to each instance in the batch separately, and
returning a tensor of identical shape - just like our (2, 4) \* (1, 4)
example above returned a tensor of shape (2, 4).

The rules for broadcasting are:

-   Each tensor must have at least one dimension - no empty tensors.
-   Comparing the dimension sizes of the two tensors, *going from last
    to first:*
    -   Each dimension must be equal, *or*
    -   One of the dimensions must be of size 1, *or*
    -   The dimension does not exist in one of the tensors

Tensors of identical shape, of course, are trivially "broadcastable", as
you saw earlier.

Here are some examples of situations that honor the above rules and
allow broadcasting:


In [32]:
torch.manual_seed(1234)  # Fija la semilla para que los números aleatorios sean reproducibles.

a = torch.ones(4, 3, 2)  # Crea un tensor base de forma (4, 3, 2) lleno de unos.
print(a)
b = a * torch.rand(3, 2)  # Broadcasting: el tensor (3,2) se replica sobre las 4 capas de 'a'.
print(b)  # Muestra el resultado de la multiplicación elemento a elemento.

c = a * torch.rand(3, 1)  # Broadcasting: la última dimensión 1 se expande a 2 columnas.
print(c)  # Muestra el resultado; cada fila repite valor en sus 2 columnas.

d = a * torch.rand(1, 2)  # Broadcasting: la primera dimensión 1 se expande en filas y capas.
print(d)  # Muestra el resultado; cada fila comparte el mismo par de valores por capa.

tensor([[[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]]])
tensor([[[0.0290, 0.4019],
         [0.2598, 0.3666],
         [0.0583, 0.7006]],

        [[0.0290, 0.4019],
         [0.2598, 0.3666],
         [0.0583, 0.7006]],

        [[0.0290, 0.4019],
         [0.2598, 0.3666],
         [0.0583, 0.7006]],

        [[0.0290, 0.4019],
         [0.2598, 0.3666],
         [0.0583, 0.7006]]])
tensor([[[0.0518, 0.0518],
         [0.4681, 0.4681],
         [0.6738, 0.6738]],

        [[0.0518, 0.0518],
         [0.4681, 0.4681],
         [0.6738, 0.6738]],

        [[0.0518, 0.0518],
         [0.4681, 0.4681],
         [0.6738, 0.6738]],

        [[0.0518, 0.0518],
         [0.4681, 0.4681],
         [0.6738, 0.6738]]])
tensor([[[0.3315, 0.7837],
         [0.3315, 0.7837],
         [0.3315, 0.7837]],

        [[0.3315,

Look closely at the values of each tensor above:

-   The multiplication operation that created `b` was broadcast over
    every "layer" of `a`.
-   For `c`, the operation was broadcast over every layer and row of
    `a` - every 3-element column is identical.
-   For `d`, we switched it around - now every *row* is identical,
    across layers and columns.

For more information on broadcasting, see the [PyTorch
documentation](https://pytorch.org/docs/stable/notes/broadcasting.html)
on the topic.

Here are some examples of attempts at broadcasting that will fail:

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>The following cell throws a run-time error. This is intentional.<pre><code>a =     torch.ones(4, 3, 2)</p>
<p>b = a * torch.rand(4, 3)    # dimensions must match last-to-first</p>
<p>c = a * torch.rand(   2, 3) # both 3rd &amp; 2nd dims different</p>
<p>d = a * torch.rand((0, ))   # can&#x27;t broadcast with an empty tensor</code></pre></p>

</div>



More Math with Tensors
======================

PyTorch tensors have over three hundred operations that can be performed
on them.

Here is a small sample from some of the major categories of operations:


In [39]:
# Funciones comunes sobre tensores
a = torch.rand(2, 4) * 2 - 1  # Crea un tensor 2x4 con valores aleatorios en el rango [-1, 1).

print('Tensor original:\n')
print(a,"\n")

print(f"Valor absoluto elemento a elemento:")
print(torch.abs(a),"\n")  # Valor absoluto elemento a elemento.

print(f"Redondeo hacia arriba (techo) elemento a elemento.")
print(torch.ceil(a),"\n")  # Redondeo hacia arriba (techo) elemento a elemento.

print(f"Redondeo hacia abajo (suelo) elemento a elemento.")
print(torch.floor(a),"\n")  # Redondeo hacia abajo (suelo) elemento a elemento.

print(f"Limita cada valor al intervalo [-0.5, 0.5].")
print(torch.clamp(a, -0.5, 0.5))  # Limita cada valor al intervalo [-0.5, 0.5]. Si el valor no está en ese intervalo lo cambia al limite del intervalo, según sea positivo o negativo

Tensor original:

tensor([[-0.1099, -0.0068,  0.5731,  0.3208],
        [-0.7394, -0.3004, -0.2352,  0.6086]]) 

Valor absoluto elemento a elemento:
tensor([[0.1099, 0.0068, 0.5731, 0.3208],
        [0.7394, 0.3004, 0.2352, 0.6086]]) 

Redondeo hacia arriba (techo) elemento a elemento.
tensor([[-0., -0., 1., 1.],
        [-0., -0., -0., 1.]]) 

Redondeo hacia abajo (suelo) elemento a elemento.
tensor([[-1., -1.,  0.,  0.],
        [-1., -1., -1.,  0.]]) 

Limita cada valor al intervalo [-0.5, 0.5].
tensor([[-0.1099, -0.0068,  0.5000,  0.3208],
        [-0.5000, -0.3004, -0.2352,  0.5000]])


In [40]:
# Funciones trigonométricas y sus inversas
angles = torch.tensor([0, math.pi / 4, math.pi / 2, 3 * math.pi / 4])  # Ángulos de ejemplo en radianes.
sines = torch.sin(angles)  # Seno de cada ángulo.
inverses = torch.asin(sines)  # Arcoseno de los senos (recupera ángulos dentro del rango principal).
print('\nSine and arcsine:')
print(angles)  # Muestra los ángulos originales.
print(sines)  # Muestra sus valores de seno.
print(inverses)  # Muestra los ángulos devueltos por arcsin.


Sine and arcsine:
tensor([0.0000, 0.7854, 1.5708, 2.3562])
tensor([0.0000, 0.7071, 1.0000, 0.7071])
tensor([0.0000, 0.7854, 1.5708, 0.7854])


In [ ]:
# Operaciones bit a bit
print('\nBitwise XOR:')
b = torch.tensor([1, 5, 11])    # Primer tensor entero.
c = torch.tensor([2, 7, 10])    # Segundo tensor entero.
print(torch.bitwise_xor(b, c))  # XOR bit a bit entre b y c.


Bitwise XOR:
tensor([3, 2, 1])


In [ ]:
# Comparaciones
print('\nBroadcasted, element-wise equality comparison:')
d = torch.tensor([[1., 2.], [3., 4.]])  # Tensor 2x2 de flotantes.
e = torch.ones(1, 2)  # Tensor 1x2 de unos; se expandirá por broadcasting.
print(torch.eq(d, e))  # Compara igualdad elemento a elemento; devuelve tensor booleano.


Broadcasted, element-wise equality comparison:
tensor([[ True, False],
        [False, False]])


In [44]:
# Reducciones
print('\nReduction ops:')
print(d)
print(torch.max(d))  # Máximo global de d (tensor escalar).
print(torch.max(d).item())  # Extrae el valor Python del tensor escalar.
print(torch.mean(d))  # Media de todos los elementos.
print(torch.std(d))  # Desviación estándar de todos los elementos.
print(torch.prod(d))  # Producto de todos los elementos.

print(torch.unique(torch.tensor([1, 2, 1, 2, 1, 2])))  # Elimina repetidos y devuelve valores únicos.


Reduction ops:
tensor([[1., 2.],
        [3., 4.]])
tensor(4.)
4.0
tensor(2.5000)
tensor(1.2910)
tensor(24.)
tensor([1, 2])


In [46]:
# Operaciones vectoriales y de álgebra lineal
v1 = torch.tensor([1., 0., 0.])  # Vector unitario en el eje x.
v2 = torch.tensor([0., 1., 0.])  # Vector unitario en el eje y.
m1 = torch.rand(2, 2)  # Matriz aleatoria 2x2.
m2 = torch.tensor([[3., 0.], [0., 3.]])  # 3 veces la matriz identidad 2x2.


print('\nVectors & Matrices:')
print(torch.linalg.cross(v2, v1))  # Producto vectorial v2 x v1 (dirección -z).
print(m1)  # Muestra la matriz aleatoria.
m3 = torch.linalg.matmul(m1, m2)  # Multiplicación matricial: escala m1 por 3.
print(m3)  # Resultado de la multiplicación matricial.
print(torch.linalg.svd(m3))  # Descomposición en valores singulares de m3.


Vectors & Matrices:
tensor([ 0.,  0., -1.])
tensor([[0.3769, 0.0108],
        [0.9455, 0.7661]])
tensor([[1.1307, 0.0323],
        [2.8365, 2.2983]])
torch.return_types.linalg_svd(
U=tensor([[-0.2468, -0.9691],
        [-0.9691,  0.2468]]),
S=tensor([3.7635, 0.6661]),
Vh=tensor([[-0.8045, -0.5939],
        [-0.5939,  0.8045]]))


This is a small sample of operations. For more details and the full
inventory of math functions, have a look at the
[documentation](https://pytorch.org/docs/stable/torch.html#math-operations).
For more details and the full inventory of linear algebra operations,
have a look at this
[documentation](https://pytorch.org/docs/stable/linalg.html).

Altering Tensors in Place
=========================

Most binary operations on tensors will return a third, new tensor. When
we say `c = a * b` (where `a` and `b` are tensors), the new tensor `c`
will occupy a region of memory distinct from the other tensors.

There are times, though, that you may wish to alter a tensor in place
-for example, if you're doing an element-wise computation where you can
discard intermediate values. For this, most of the math functions have a
version with an appended underscore (`_`) that will alter a tensor in
place.

For example:


In [2]:
a = torch.tensor([0, math.pi / 4, math.pi / 2, 3 * math.pi / 4])
print('a:')
print(a)
print(torch.sin(a))   # this operation creates a new tensor in memory
print(a)              # a has not changed

b = torch.tensor([0, math.pi / 4, math.pi / 2, 3 * math.pi / 4])
print('\nb:')
print(b)
print(torch.sin_(b))  # note the underscore
print(b)              # b has changed

a:
tensor([0.0000, 0.7854, 1.5708, 2.3562])
tensor([0.0000, 0.7071, 1.0000, 0.7071])
tensor([0.0000, 0.7854, 1.5708, 2.3562])

b:
tensor([0.0000, 0.7854, 1.5708, 2.3562])
tensor([0.0000, 0.7071, 1.0000, 0.7071])
tensor([0.0000, 0.7071, 1.0000, 0.7071])


For arithmetic operations, there are functions that behave similarly:


In [13]:
a = torch.ones(2, 2)  # Crea un tensor 2x2 lleno de unos.
b = torch.rand(2, 2)  # Crea un tensor 2x2 con valores aleatorios en [0, 1).

print('Before:')  # Muestra el estado inicial de los tensores.
print(f"Tensor 'a':\n {a}")  # Imprime el contenido actual de a.
print(f"Tensor 'b':\n {b}")  # Imprime el contenido actual de b.

print('\nAfter adding:')  # Indica la sección después de sumar en sitio.
a.add_(b)  # Suma b a a en sitio (in-place) y devuelve a modificado.
print(f"A 'a' se le suma 'b', pero 'b' no cambia porque la operación es in-place sobre 'a':")
print(f"Tensor 'a':\n {a}")  # Verifica que a cambió porque la operación fue in-place.
print(f"Tensor 'b':\n {b}")  # b no cambia durante a.add_(b).

print('\nAfter multiplying')  # Indica la sección después de multiplicar en sitio.
print(f"A 'b' se le multiplica por sí mismo, y 'b' cambia porque la operación es in-place sobre 'b':")
b.mul_(b)  # Multiplica b por sí mismo en sitio (in-place), elemento a elemento.
print(f"Tensor 'b':\n {b}")  # Verifica que b también cambió por la operación in-place.

Before:
Tensor 'a':
 tensor([[1., 1.],
        [1., 1.]])
Tensor 'b':
 tensor([[0.1817, 0.4745],
        [0.5505, 0.5400]])

After adding:
A 'a' se le suma 'b', pero 'b' no cambia porque la operación es in-place sobre 'a':
Tensor 'a':
 tensor([[1.1817, 1.4745],
        [1.5505, 1.5400]])
Tensor 'b':
 tensor([[0.1817, 0.4745],
        [0.5505, 0.5400]])

After multiplying
A 'b' se le multiplica por sí mismo, y 'b' cambia porque la operación es in-place sobre 'b':
Tensor 'b':
 tensor([[0.0330, 0.2252],
        [0.3031, 0.2916]])


Note that these in-place arithmetic functions are methods on the
`torch.Tensor` object, not attached to the `torch` module like many
other functions (e.g., `torch.sin()`). As you can see from `a.add_(b)`,
*the calling tensor is the one that gets changed in place.*

There is another option for placing the result of a computation in an
existing, allocated tensor. Many of the methods and functions we've seen
so far - including creation methods! - have an `out` argument that lets
you specify a tensor to receive the output. If the `out` tensor is the
correct shape and `dtype`, this can happen without a new memory
allocation:


In [19]:
a = torch.rand(2, 2)  # Tensor 2x2 aleatorio que actuará como primer operando de la multiplicación matricial.
b = torch.rand(2, 2)  # Tensor 2x2 aleatorio que actuará como segundo operando.
c = torch.zeros(2, 2)  # Tensor destino preasignado; aquí se escribirá el resultado usando out=.
old_id = id(c)  # Guardamos la identidad de memoria de c para comprobar que no se reemplaza el objeto.
print(f"Tipo de old_id: {type(old_id)}, valor de old_id: {old_id}")  # Verifica que old_id es un entero.
print(f"Tensor c:\n {c}")  # Estado inicial de c (todo ceros).

d = torch.matmul(a, b, out=c)  # Calcula a @ b y coloca el resultado directamente dentro de c.
print(f"\nTensor c después de matmul a y b:\n {c}")  # c cambió su contenido con el resultado de la operación.

assert c is d  # Verifica que d y c son exactamente el mismo objeto en memoria.
assert id(c) == old_id  # Confirma que c conserva su identidad original tras la operación.

torch.rand(2, 2, out=c)  # También en creación: rellena c con nuevos valores aleatorios sin crear otro tensor.
print(f"\nTensor c después de rellenar con valores aleatorios:\n {c}")  # c vuelve a cambiar, ahora con números aleatorios nuevos.
assert id(c) == old_id  # c sigue siendo el mismo objeto: solo cambió su contenido.

Tipo de old_id: <class 'int'>, valor de old_id: 139851801706272
Tensor c:
 tensor([[0., 0.],
        [0., 0.]])

Tensor c después de matmul a y b:
 tensor([[0.1437, 0.0479],
        [0.2862, 0.2826]])

Tensor c después de rellenar con valores aleatorios:
 tensor([[0.3066, 0.7658],
        [0.4217, 0.2061]])


Copying Tensors
===============

As with any object in Python, assigning a tensor to a variable makes the
variable a *label* of the tensor, and does not copy it. For example:


In [23]:
"""
                                    --- IMPORTANTE ---
"""
a = torch.ones(2, 2)  # Crea un tensor 2x2 inicializado con unos.
b = a  # b no copia el tensor: solo apunta al mismo objeto que a.
print(f"Tensor 'a':\n {a}") 
print(f"Tensor 'b':\n {b}")  # a y b muestran lo mismo porque son el mismo tensor.

print(f"\nModificamos un elemento de 'a' y vemos que 'b' también cambia porque ambos apuntan al mismo tensor:")
a[0][1] = 561  # Modificamos un elemento de a en sitio (fila 0, columna 1).
print(f"Tensor 'a':\n {a}")  # Muestra a con el cambio aplicado.
print(f"Tensor 'b':\n {b}")       # Como b referencia el mismo tensor, refleja el cambio inmediatamente.

Tensor 'a':
 tensor([[1., 1.],
        [1., 1.]])
Tensor 'b':
 tensor([[1., 1.],
        [1., 1.]])

Modificamos un elemento de 'a' y vemos que 'b' también cambia porque ambos apuntan al mismo tensor:
Tensor 'a':
 tensor([[  1., 561.],
        [  1.,   1.]])
Tensor 'b':
 tensor([[  1., 561.],
        [  1.,   1.]])


But what if you want a separate copy of the data to work on? The
`clone()` method is there for you:


In [26]:
"""
                                    --- IMPORTANTE ---
"""
a = torch.ones(2, 2)  # Crea un tensor 2x2 lleno de unos.
print(f"Tensor 'a':\n {a}")
b = a.clone()  # Crea una copia real de los datos en un objeto distinto.
print(f"Después de clonar 'a' en 'b', ambos tienen los mismos valores pero son objetos diferentes en memoria:")
print(f"Tensor 'a':\n {a}")
print(f"Tensor 'b' (clon de 'a'):\n {b}")  # b muestra los mismos valores que a inicialmente, pero es un objeto diferente.

print(f"\nVerificamos que 'a' y 'b' son objetos distintos en memoria, pero con los mismos valores:")
assert b is not a  # Comprueba que b y a NO son el mismo objeto en memoria.
print(torch.eq(a, b))  # Verifica que, aunque son objetos distintos, sus valores iniciales coinciden.

print(f"\nModificamos un elemento de 'a' y vemos que 'b' no cambia porque es una copia independiente:")
a[0][1] = 561  # Modifica un elemento de a después de clonar.
print(f"Tensor 'a' después de la modificación:\n {a}")  # Muestra a con el cambio aplicado.
print(f"Tensor 'b' después de la modificación de 'a':\n {b}")  # b no cambia porque es una copia independiente de a.

Tensor 'a':
 tensor([[1., 1.],
        [1., 1.]])
Después de clonar 'a' en 'b', ambos tienen los mismos valores pero son objetos diferentes en memoria:
Tensor 'a':
 tensor([[1., 1.],
        [1., 1.]])
Tensor 'b' (clon de 'a'):
 tensor([[1., 1.],
        [1., 1.]])

Verificamos que 'a' y 'b' son objetos distintos en memoria, pero con los mismos valores:
tensor([[True, True],
        [True, True]])

Modificamos un elemento de 'a' y vemos que 'b' no cambia porque es una copia independiente:
Tensor 'a' después de la modificación:
 tensor([[  1., 561.],
        [  1.,   1.]])
Tensor 'b' después de la modificación de 'a':
 tensor([[1., 1.],
        [1., 1.]])


**There is an important thing to be aware of when using
\`\`clone()\`\`.** If your source tensor has autograd, enabled then so
will the clone. **This will be covered more deeply in the video on
autograd,** but if you want the light version of the details, continue
on.

*In many cases, this will be what you want.* For example, if your model
has multiple computation paths in its `forward()` method, and *both* the
original tensor and its clone contribute to the model's output, then to
enable model learning you want autograd turned on for both tensors. If
your source tensor has autograd enabled (which it generally will if it's
a set of learning weights or derived from a computation involving the
weights), then you'll get the result you want.

On the other hand, if you're doing a computation where *neither* the
original tensor nor its clone need to track gradients, then as long as
the source tensor has autograd turned off, you're good to go.

*There is a third case,* though: Imagine you're performing a computation
in your model's `forward()` function, where gradients are turned on for
everything by default, but you want to pull out some values mid-stream
to generate some metrics. In this case, you *don't* want the cloned copy
of your source tensor to track gradients - performance is improved with
autograd's history tracking turned off. For this, you can use the
`.detach()` method on the source tensor:


In [27]:
a = torch.rand(2, 2, requires_grad=True)  # Crea un tensor aleatorio con autograd activado para rastrear gradientes.
print(a)  # Muestra a; debería indicar que requiere gradiente.

b = a.clone()  # Clona a manteniendo su vínculo con autograd (también rastrea gradientes).
print(b)  # Muestra b; conserva valores y contexto de gradiente de a.

c = a.detach().clone()  # Separa a del grafo de autograd y luego clona: c queda sin seguimiento de gradientes.
print(c)  # Muestra c; ya no participa en el cálculo de gradientes.

print(a)  # a permanece sin cambios y sigue con requires_grad=True.

tensor([[0.7437, 0.0562],
        [0.8646, 0.5295]], requires_grad=True)
tensor([[0.7437, 0.0562],
        [0.8646, 0.5295]], grad_fn=<CloneBackward0>)
tensor([[0.7437, 0.0562],
        [0.8646, 0.5295]])
tensor([[0.7437, 0.0562],
        [0.8646, 0.5295]], requires_grad=True)


What's happening here?

-   We create `a` with `requires_grad=True` turned on. **We haven't
    covered this optional argument yet, but will during the unit on
    autograd.**
-   When we print `a`, it informs us that the property
    `requires_grad=True` - this means that autograd and computation
    history tracking are turned on.
-   We clone `a` and label it `b`. When we print `b`, we can see that
    it's tracking its computation history - it has inherited `a`'s
    autograd settings, and added to the computation history.
-   We clone `a` into `c`, but we call `detach()` first.
-   Printing `c`, we see no computation history, and no
    `requires_grad=True`.

The `detach()` method *detaches the tensor from its computation
history.* It says, "do whatever comes next as if autograd was off." It
does this *without* changing `a` - you can see that when we print `a`
again at the end, it retains its `requires_grad=True` property.

Moving to
[Accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
\-\-\-\-\-\-\-\-\-\-\-\--

One of the major advantages of PyTorch is its robust acceleration on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. So far, everything we've done has been
on CPU. How do we move to the faster hardware?

First, we should check whether an accelerator is available, with the
`is_available()` method.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>If you do not have an accelerator, the executable cells in this section will not execute anyaccelerator-related code.</p>

</div>



In [28]:
if torch.accelerator.is_available():  # Comprueba si hay un acelerador disponible (GPU u otro backend soportado).
    print('We have an accelerator!')  # Se ejecuta cuando PyTorch detecta hardware acelerado.
else:  # Rama alternativa cuando no hay acelerador disponible.
    print('Sorry, CPU only.')  # Informa que solo se usará CPU.

Sorry, CPU only.


Once we've determined that one or more accelerators is available, we
need to put our data someplace where the accelerator can see it. Your
CPU does computation on data in your computer's RAM. Your accelerator
has dedicated memory attached to it. Whenever you want to perform a
computation on a device, you must move *all* the data needed for that
computation to memory accessible by that device. (Colloquially, "moving
the data to memory accessible by the GPU" is shorted to, "moving the
data to the GPU".)

There are multiple ways to get your data onto your target device. You
may do it at creation time:


In [29]:
if torch.accelerator.is_available():  # Verifica si hay un acelerador disponible para ejecutar operaciones más rápido.
    gpu_rand = torch.rand(2, 2, device=torch.accelerator.current_accelerator())  # Crea el tensor directamente en el dispositivo acelerado activo.
    print(gpu_rand)  # Muestra el tensor; en la salida se indica también el dispositivo donde fue creado.
else:  # Si no hay acelerador, se ejecuta esta rama alternativa.
    print('Sorry, CPU only.')  # Informa que en este entorno solo está disponible la CPU.

Sorry, CPU only.


By default, new tensors are created on the CPU, so we have to specify
when we want to create our tensor on the accelerator with the optional
`device` argument. You can see when we print the new tensor, PyTorch
informs us which device it's on (if it's not on CPU).

You can query the number of accelerators with
`torch.accelerator.device_count()`. If you have more than one
accelerator, you can specify them by index, take CUDA for example:
`device='cuda:0'`, `device='cuda:1'`, etc.

As a coding practice, specifying our devices everywhere with string
constants is pretty fragile. In an ideal world, your code would perform
robustly whether you're on CPU or accelerator hardware. You can do this
by creating a device handle that can be passed to your tensors instead
of a string:


In [30]:
my_device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')  # Selecciona el acelerador actual si existe; si no, usa CPU.
print('Device: {}'.format(my_device))  # Muestra el dispositivo elegido para trabajar.

x = torch.rand(2, 2, device=my_device)  # Crea un tensor 2x2 directamente en el dispositivo seleccionado.
print(x)  # Imprime el tensor (y su dispositivo si no está en CPU).

Device: cpu
tensor([[0.2819, 0.6537],
        [0.2057, 0.2538]])


If you have an existing tensor living on one device, you can move it to
another with the `to()` method. The following line of code creates a
tensor on CPU, and moves it to whichever device handle you acquired in
the previous cell.


In [34]:
y = torch.rand(2, 2)  # Crea un tensor 2x2 en CPU (comportamiento por defecto).
y = y.to(my_device)  # Mueve el tensor y al dispositivo seleccionado en my_device (acelerador o CPU).
print('Device: {}'.format(my_device))  # Muestra el dispositivo elegido para trabajar.
print(y.device)         
print(y.device.type)

Device: cpu
cpu
cpu


It is important to know that in order to do computation involving two or
more tensors, *all of the tensors must be on the same device*. The
following code will throw a runtime error, regardless of whether you
have an accelerator device available, take CUDA for example:

``` {.python}
x = torch.rand(2, 2)
y = torch.rand(2, 2, device='cuda')
z = x + y  # exception will be thrown
```


Manipulating Tensor Shapes
==========================

Sometimes, you'll need to change the shape of your tensor. Below, we'll
look at a few common cases, and how to handle them.

Changing the Number of Dimensions
---------------------------------

One case where you might need to change the number of dimensions is
passing a single instance of input to your model. PyTorch models
generally expect *batches* of input.

For example, imagine having a model that works on 3 x 226 x 226 images
-a 226-pixel square with 3 color channels. When you load and transform
it, you'll get a tensor of shape `(3, 226, 226)`. Your model, though, is
expecting input of shape `(N, 3, 226, 226)`, where `N` is the number of
images in the batch. So how do you make a batch of one?


In [35]:
a = torch.rand(3, 226, 226)  # Simula una imagen individual con forma (canales, alto, ancho).

# a.unsqueeze(0), que crea otro tensor (normalmente una vista) con una dimensión extra.
# Por eso, a continuación a.shape es torch.Size([3, 226, 226]) y b.shape es torch.Size([1, 3, 226, 226]).
b = a.unsqueeze(0)  # Inserta una dimensión de tamaño 1 al inicio (posicion 0) para crear batch de tamaño 1: (1, 3, 226, 226).

print(a.shape)  # Forma original sin dimensión de batch.
print(b.shape)  # Nueva forma con dimensión de batch añadida.

torch.Size([3, 226, 226])
torch.Size([1, 3, 226, 226])


The `unsqueeze()` method adds a dimension of extent 1. `unsqueeze(0)`
adds it as a new zeroth dimension - now you have a batch of one!

So if that's *un*squeezing? What do we mean by squeezing? We're taking
advantage of the fact that any dimension of extent 1 *does not* change
the number of elements in the tensor.


In [36]:
c = torch.rand(1, 1, 1, 1, 1)  # Crea un tensor de 5 dimensiones, cada una de tamaño 1.
print(c)  # Imprime el tensor; se observan muchos corchetes por la cantidad de dimensiones.

tensor([[[[[0.6875]]]]])


Continuing the example above, let's say the model's output is a
20-element vector for each input. You would then expect the output to
have shape `(N, 20)`, where `N` is the number of instances in the input
batch. That means that for our single-input batch, we'll get an output
of shape `(1, 20)`.

What if you want to do some *non-batched* computation with that output
-something that's just expecting a 20-element vector?


In [37]:
a = torch.rand(1, 20)  # Simula una salida batcheada con forma (1, 20): 1 elemento en el batch y 20 características. Hay 2 dimesiones (2D), pero la primera es de tamaño 1.
print(a.shape)  # Muestra la forma original con dimensión extra de batch.
print(a)  # Al imprimirlo, se observan corchetes adicionales por la dimensión de tamaño 1.

"""
                    Funcionamiento de squeeze(dim):

Piensa en la forma del tensor como una lista de tamaños, por ejemplo (1, 20).

dim es qué posición de esa lista quieres revisar.
squeeze(dim) mira solo esa posición.
Si en esa posición hay un 1, la elimina.
Si hay otro número (2, 3, 20…), no puede eliminarla y lo deja igual.
"""
b = a.squeeze(0)  # Elimina la dimensión 0 solo si su tamaño es 1; pasa de (1, 20) a (20,).
print(b.shape)  # Verifica que ahora es un vector 1D de 20 elementos.
print(b)  # Imprime el vector sin la dimensión de batch.

c = torch.rand(2, 2)  # Crea una matriz 2x2 (ninguna dimensión es 1).
print(c.shape)  # Confirma su forma inicial.

d = c.squeeze(0)  # Intenta eliminar la dimensión 0, pero como mide 2, no se puede comprimir.
print(d.shape)  # La forma se mantiene en (2, 2), sin cambios.

torch.Size([1, 20])
tensor([[0.1716, 0.7611, 0.7969, 0.4875, 0.6066, 0.8725, 0.9503, 0.1053, 0.6529,
         0.1540, 0.0688, 0.8364, 0.9011, 0.5638, 0.4605, 0.3545, 0.2245, 0.2248,
         0.9885, 0.1141]])
torch.Size([20])
tensor([0.1716, 0.7611, 0.7969, 0.4875, 0.6066, 0.8725, 0.9503, 0.1053, 0.6529,
        0.1540, 0.0688, 0.8364, 0.9011, 0.5638, 0.4605, 0.3545, 0.2245, 0.2248,
        0.9885, 0.1141])
torch.Size([2, 2])
torch.Size([2, 2])


You can see from the shapes that our 2-dimensional tensor is now
1-dimensional, and if you look closely at the output of the cell above
you'll see that printing `a` shows an "extra" set of square brackets
`[]` due to having an extra dimension.

You may only `squeeze()` dimensions of extent 1. See above where we try
to squeeze a dimension of size 2 in `c`, and get back the same shape we
started with. Calls to `squeeze()` and `unsqueeze()` can only act on
dimensions of extent 1 because to do otherwise would change the number
of elements in the tensor.

Another place you might use `unsqueeze()` is to ease broadcasting.
Recall the example above where we had the following code:

``` {.python}
a = torch.ones(4, 3, 2)

c = a * torch.rand(   3, 1) # 3rd dim = 1, 2nd dim identical to a
print(c)
```

The net effect of that was to broadcast the operation over dimensions 0
and 2, causing the random, 3 x 1 tensor to be multiplied element-wise by
every 3-element column in `a`.

What if the random vector had just been 3-element vector? We'd lose the
ability to do the broadcast, because the final dimensions would not
match up according to the broadcasting rules. `unsqueeze()` comes to the
rescue:


In [41]:
a = torch.ones(4, 3, 2)  # Tensor base con forma (4, 3, 2): 4 bloques, 3 filas y 2 columnas.
print(a,"\n")  # Muestra 'a' para visualizar su estructura 3D.

b = torch.rand(3)  # Vector 1D de forma (3,). Así como está, no se puede multiplicar directamente por 'a'.
print(b.shape)  # Confirma que b tiene una sola dimensión.
print(b, "\n")  # Muestra los 3 valores aleatorios de b.

c = b.unsqueeze(1)  # Inserta una dimensión en la posición 1: de (3,) pasa a (3, 1).
print(c.shape)  # Verifica la nueva forma 2D.
print(c, "\n")  # Ahora cada uno de los 3 valores queda en su propia fila.

print(f"a*c:\n {a * c}")  # Con forma (3,1), c sí se puede expandir por broadcasting y multiplicar con a.

tensor([[[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]]]) 

torch.Size([3])
tensor([0.1165, 0.5175, 0.0009]) 

torch.Size([3, 1])
tensor([[0.1165],
        [0.5175],
        [0.0009]]) 

a*c:
 tensor([[[0.1165, 0.1165],
         [0.5175, 0.5175],
         [0.0009, 0.0009]],

        [[0.1165, 0.1165],
         [0.5175, 0.5175],
         [0.0009, 0.0009]],

        [[0.1165, 0.1165],
         [0.5175, 0.5175],
         [0.0009, 0.0009]],

        [[0.1165, 0.1165],
         [0.5175, 0.5175],
         [0.0009, 0.0009]]])


The `squeeze()` and `unsqueeze()` methods also have in-place versions,
`squeeze_()` and `unsqueeze_()`:


In [42]:
batch_me = torch.rand(3, 226, 226)  # Crea un tensor 3D que representa una imagen (canales, alto, ancho).
print(batch_me.shape)  # Muestra la forma inicial: (3, 226, 226).

batch_me.unsqueeze_(0)  # Versión in-place de unsqueeze: inserta dimensión 1 en la posición 0 y modifica el mismo tensor.
print(batch_me.shape)  # Nueva forma: (1, 3, 226, 226), ahora incluye dimensión de batch.

torch.Size([3, 226, 226])
torch.Size([1, 3, 226, 226])


Sometimes you'll want to change the shape of a tensor more radically,
while still preserving the number of elements and their contents. One
case where this happens is at the interface between a convolutional
layer of a model and a linear layer of the model - this is common in
image classification models. A convolution kernel will yield an output
tensor of shape *features x width x height,* but the following linear
layer expects a 1-dimensional input. `reshape()` will do this for you,
provided that the dimensions you request yield the same number of
elements as the input tensor has:


In [43]:
output3d = torch.rand(6, 20, 20)  # Crea un tensor 3D: 6 bloques de tamaño 20x20.
print(output3d.shape)  # Muestra la forma original: (6, 20, 20).

input1d = output3d.reshape(6 * 20 * 20)  # Reorganiza todos los elementos en un vector 1D de longitud 2400.
print(input1d.shape)  # Verifica la nueva forma: (2400,).

# También se puede usar la versión funcional desde el módulo torch.
print(torch.reshape(output3d, (6 * 20 * 20,)).shape)  # Misma operación de reshape; mismo resultado de forma (2400,).

torch.Size([6, 20, 20])
torch.Size([2400])
torch.Size([2400])


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>The <code>(6 * 20 * 20,)</code> argument in the final line of the cellabove is because PyTorch expects a  when specifying atensor shape - but when the shape is the first argument of a method, itlets us cheat and just use a series of integers. Here, we had to add theparentheses and comma to convince the method that this is really aone-element tuple.</p>

</div>

When it can, `reshape()` will return a *view* on the tensor to be
changed - that is, a separate tensor object looking at the same
underlying region of memory. *This is important:* That means any change
made to the source tensor will be reflected in the view on that tensor,
unless you `clone()` it.

There *are* conditions, beyond the scope of this introduction, where
`reshape()` has to return a tensor carrying a copy of the data. For more
information, see the
[docs](https://pytorch.org/docs/stable/torch.html#torch.reshape).


NumPy Bridge
============

In the section above on broadcasting, it was mentioned that PyTorch's
broadcast semantics are compatible with NumPy's - but the kinship
between PyTorch and NumPy goes even deeper than that.

If you have existing ML or scientific code with data stored in NumPy
ndarrays, you may wish to express that same data as PyTorch tensors,
whether to take advantage of PyTorch's GPU acceleration, or its
efficient abstractions for building ML models. It's easy to switch
between ndarrays and PyTorch tensors:


In [44]:
import numpy as np  # Importa NumPy para crear y manipular arreglos tipo ndarray.

numpy_array = np.ones((2, 3))  # Crea un arreglo NumPy de forma (2, 3) lleno de unos.
print(numpy_array)  # Muestra el arreglo original en formato NumPy.

pytorch_tensor = torch.from_numpy(numpy_array)  # Convierte el ndarray a tensor de PyTorch sin copiar datos.
print(pytorch_tensor)  # Muestra el tensor resultante con la misma forma y valores.

[[1. 1. 1.]
 [1. 1. 1.]]
tensor([[1., 1., 1.],
        [1., 1., 1.]], dtype=torch.float64)


PyTorch creates a tensor of the same shape and containing the same data
as the NumPy array, going so far as to keep NumPy's default 64-bit float
data type.

The conversion can just as easily go the other way:


In [45]:
pytorch_rand = torch.rand(2, 3)  # Crea un tensor de PyTorch de forma (2, 3) con valores aleatorios en [0, 1).
print(pytorch_rand)  # Muestra el tensor original en formato PyTorch.

numpy_rand = pytorch_rand.numpy()  # Convierte el tensor a ndarray de NumPy (normalmente compartiendo memoria en CPU).
print(numpy_rand)  # Muestra el arreglo equivalente en formato NumPy.

tensor([[0.4455, 0.8993, 0.9929],
        [0.8771, 0.2327, 0.6177]])
[[0.44550127 0.899323   0.99293983]
 [0.87709564 0.23273051 0.61766845]]


It is important to know that these converted objects are using *the same
underlying memory* as their source objects, meaning that changes to one
are reflected in the other:


In [46]:
"""
                ---     OJO CON LA MEMORIA COMPARTIDA ENTRE NUMPY Y PYTORCH     ---
                
Cuando conviertes entre NumPy y PyTorch usando torch.from_numpy() o tensor.numpy(), ambos objetos comparten la misma memoria subyacente (si están en CPU). Esto significa que si modificas el contenido de uno, el cambio se refleja automáticamente en el otro, porque ambos apuntan a la misma ubicación de memoria. Ten cuidado al modificar los datos, ya que puede afectar a ambos objetos sin que te des cuenta.
"""
numpy_array[1, 1] = 23  # Modifica un elemento del ndarray original de NumPy.
print(pytorch_tensor)  # Se refleja también en el tensor de PyTorch porque comparten la misma memoria.

pytorch_rand[1, 1] = 17  # Modifica un elemento del tensor de PyTorch.
print(numpy_rand)  # El cambio aparece en el ndarray de NumPy por la memoria compartida.

tensor([[ 1.,  1.,  1.],
        [ 1., 23.,  1.]], dtype=torch.float64)
[[ 0.44550127  0.899323    0.99293983]
 [ 0.87709564 17.          0.61766845]]


In [ ]:
"""
                        --- EJEMPLO: CONVERSIÓN SIN MEMORIA COMPARTIDA ---
Aquí sí copiamos los datos, así que cambiar NumPy no afecta a PyTorch y viceversa.
"""

# NumPy -> PyTorch sin compartir memoria
numpy_array_no_share = np.ones((2, 3))
pytorch_tensor_no_share = torch.tensor(numpy_array_no_share.copy())

numpy_array_no_share[1, 1] = 23
print("NumPy modificado:\n", numpy_array_no_share)
print("Tensor PyTorch (no debe cambiar por el cambio en NumPy):\n", pytorch_tensor_no_share)

# PyTorch -> NumPy sin compartir memoria
pytorch_rand_no_share = torch.rand(2, 3)
numpy_rand_no_share = pytorch_rand_no_share.detach().cpu().numpy().copy()
# Otra forma: numpy_rand_no_share = np.array(pytorch_rand_no_share.detach().cpu(), copy=True)

pytorch_rand_no_share[1, 1] = 17
print("\nTensor PyTorch modificado:\n", pytorch_rand_no_share)
print("NumPy (no debe cambiar por el cambio en PyTorch):\n", numpy_rand_no_share)

NumPy modificado:
 [[ 1.  1.  1.]
 [ 1. 23.  1.]]
Tensor PyTorch (no debe cambiar por el cambio en NumPy):
 tensor([[1., 1., 1.],
        [1., 1., 1.]], dtype=torch.float64)

Tensor PyTorch modificado:
 tensor([[ 0.1226,  0.4393,  0.9722],
        [ 0.6601, 17.0000,  0.4472]])
NumPy (no debe cambiar por el cambio en PyTorch):
 [[0.12261999 0.43930024 0.9722191 ]
 [0.6600757  0.981081   0.4472158 ]]
